# rotation-matrix-3d — ex1: Rodrigues rotation about an arbitrary 3-D axis

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `rotation-matrix-3d`. Running the final beacon cell reports progress against the `Geometry: Rotation matrix 3-D (full)` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt
import math

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Geometry: Rotation matrix 3-D (full)` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`rotation-matrix-3d`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "rotation-matrix-3d"
DD_SUBTOPIC = "Geometry: Rotation matrix 3-D (full)"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## General 3-D rotation (Rodrigues' formula) — quick refresher

**The matrix.** Right-hand rotation by `θ` about a UNIT axis `k = (kx, ky, kz)`:
```
R = I + sin(θ) * K + (1 - cos(θ)) * K²
```
where `K` is the skew-symmetric cross-product matrix of `k`:
```
K = [[ 0,  -kz,  ky],
     [ kz,  0,  -kx],
     [-ky,  kx,  0]]
```
**Special cases.** Axis = X gives `R_x(θ)`, axis = Y gives `R_y(θ)`, axis = Z gives `R_z(θ)`. The previous drill (`rotation-matrix-3d-y-axis`) is the Y-axis special case.

**Axis must be unit length.** If `||k|| != 1`, Rodrigues' formula returns a scaled rotation — wrong. Always `k = k / k.norm()` before use.

**Verifying orthogonality.** `R @ R.T == I` and `det(R) == +1`. Use these as numerical sanity checks (tolerance ~1e-6).

### Exercise 1 — Rodrigues rotation about an arbitrary 3-D axis

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Apply
> LO: Apply Rodrigues' formula `R = I + sinθ K + (1-cosθ) K²` to construct the 3×3 rotation matrix for an arbitrary unit-axis + angle, with a 3-D scatter showing rotated points.
> Keywords: rotation, rodrigues, skew-symmetric, axis-angle, 3D
> ```

**KCs targeted:** `build-skew-symmetric-matrix`, `rodrigues-formula-assemble`

Implement `ex1_rot3d(axis, theta)`. Inputs:
- `axis`: `(3,)` float tensor (NOT assumed unit — normalize inside).
- `theta`: scalar float angle in radians.

Steps:
1. Normalize: `k = axis / axis.norm()`.
2. Build the skew-symmetric `K`:
   ```python
   K = t.tensor([[    0, -k[2],  k[1]],
                 [ k[2],     0, -k[0]],
                 [-k[1],  k[0],    0]])
   ```
   (Use `k[0].item()` etc. if you build via `t.tensor`, OR use `t.stack` / `t.zeros` to keep the tensor backend.)
3. Apply Rodrigues:
   ```python
   R = t.eye(3) + math.sin(theta) * K + (1 - math.cos(theta)) * (K @ K)
   ```
4. Return `R` as a `(3, 3)` `float32` tensor.

**Sanity checks** (asserted in the test, but worth running yourself):
- `R @ R.T ≈ I`
- `det(R) ≈ +1`
- Rotating `k` itself gives back `k` (the axis is fixed).

In [ ]:
def ex1_rot3d(axis: Tensor, theta: float) -> Tensor:
    k = axis / axis.norm()
    kx, ky, kz = k[0], k[1], k[2]
    K = t.stack([
        t.stack([t.zeros_like(kx),          -kz,                  ky]),
        t.stack([                 kz, t.zeros_like(kx),          -kx]),
        t.stack([                -ky,                kx, t.zeros_like(kx)]),
    ])
    s, c = math.sin(theta), math.cos(theta)
    return t.eye(3) + s * K + (1 - c) * (K @ K)


<details><summary>Solution</summary>

```python
def ex1_rot3d(axis: Tensor, theta: float) -> Tensor:
    k = axis / axis.norm()
    kx, ky, kz = k[0], k[1], k[2]
    K = t.stack([
        t.stack([t.zeros_like(kx),          -kz,                  ky]),
        t.stack([                 kz, t.zeros_like(kx),          -kx]),
        t.stack([                -ky,                kx, t.zeros_like(kx)]),
    ])
    s, c = math.sin(theta), math.cos(theta)
    return t.eye(3) + s * K + (1 - c) * (K @ K)
```

**Why normalize the axis.** Rodrigues' formula assumes `||k||=1`. If you pass `(1, 2, 3)` directly, `K @ K` scales by `||k||^2 = 14`, and you get a wildly non-orthogonal matrix. Always `k = axis / axis.norm()` first.

**Why `K @ K` not `K^2` via element-wise square.** `K**2` (Python) does element-wise squaring on tensors — that's NOT the matrix square. Use `K @ K` (matmul) or `t.linalg.matrix_power(K, 2)`.

**Why the cube viz.** Single-axis rotations (X/Y/Z) keep one axis fixed, which doesn't visually exercise the off-diagonal Rodrigues terms. Rotating about the body diagonal `(1, 1, 1)` involves ALL components of `K` — a more interesting test of the formula.

**Generalizing to a batch.** For a stack of `(B, 3)` axes and `(B,)` angles, broadcast `K` along the batch dim. PyTorch's `linalg` ops handle this natively — useful for robotics IK and rigid-body sims.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()